# radiative transfer
libradtran surface irradiance for smoke and stratospheric sulfate added to the clean sky aerosol, by added aod and solar zenith angle (figure 3). smoke and clean aerosol optics are medians of the aeronet values in the site samples; sulfate optics from a mie calculation

In [1]:
from pathlib import Path
import os, subprocess
from concurrent.futures import ThreadPoolExecutor
import numpy as np, pandas as pd

data, out = Path("data"), Path("results")
out.mkdir(exist_ok=True)
# libradtran 2.0.6 install, run through wsl on windows
shell = ["wsl", "-d", "Ubuntu", "-e", "bash", "-c"] if os.name == "nt" else ["bash", "-c"]
lrt = subprocess.run(shell + ["cd ~/lrt/libRadtran-2.0.6 && pwd"], capture_output=True, text=True).stdout.strip()
sites = ["US-Bi2", "US-Ne1", "US-Ne2", "US-Ne3", "US-xUN", "US-UMB", "US-MMS", "US-Ha1"]
aods = [0, 0.02, 0.05, 0.1, 0.15, 0.2, 0.3, 0.4, 0.6, 0.8, 1.0, 1.3, 1.6, 2.0]
szas = [20, 30, 40, 50, 60]

In [2]:
# clean sky and smoke aerosol: medians over all clean reference and all smoke steps of the eight site samples
s = pd.concat([pd.read_csv(data / f"site_{x}.csv") for x in sites])
med = lambda d: {"ssa": round(d.ssa_sw.median(), 3), "g": round(d.g_sw.median(), 3), "ae": round(d.AE.median(), 3),
                 "aod": round(d.AOD550.median(), 3)}
background, smoke = med(s[s.is_bg]), med(s[s.is_smoke])
background, smoke

Out[2]: 
({'ssa': np.float64(0.941),
  'g': np.float64(0.669),
  'ae': np.float64(1.213),
  'aod': np.float64(0.076)},
 {'ssa': np.float64(0.959),
  'g': np.float64(0.65),
  'ae': np.float64(1.685),
  'aod': np.float64(0.305)})


({'ssa': np.float64(0.941),
  'g': np.float64(0.669),
  'ae': np.float64(1.213),
  'aod': np.float64(0.076)},
 {'ssa': np.float64(0.959),
  'g': np.float64(0.65),
  'ae': np.float64(1.685),
  'aod': np.float64(0.305)})

In [3]:
def mie(m, x):
    # bohren and huffman: extinction and scattering efficiency and asymmetry for a sphere of size parameter x
    nstop = int(x + 4 * x ** (1 / 3) + 2)
    nmx = int(max(nstop, abs(m * x)) + 15)
    y = m * x
    d = np.zeros(nmx + 1, complex)
    for n in range(nmx, 0, -1):
        d[n - 1] = n / y - 1 / (d[n] + n / y)
    psi0, psi1, chi0, chi1 = np.cos(x), np.sin(x), -np.sin(x), np.cos(x)
    xi1 = complex(psi1, -chi1)
    qs = qe = gs = 0.0
    an1 = bn1 = 0
    for n in range(1, nstop + 1):
        psi, chi = (2 * n - 1) / x * psi1 - psi0, (2 * n - 1) / x * chi1 - chi0
        xi = complex(psi, -chi)
        an = ((d[n] / m + n / x) * psi - psi1) / ((d[n] / m + n / x) * xi - xi1)
        bn = ((d[n] * m + n / x) * psi - psi1) / ((d[n] * m + n / x) * xi - xi1)
        qs += (2 * n + 1) * (abs(an) ** 2 + abs(bn) ** 2)
        qe += (2 * n + 1) * (an + bn).real
        gs += (2 * n + 1) / (n * (n + 1)) * (an * np.conj(bn)).real
        if n > 1:
            gs += (n - 1) * (n + 1) / n * (an1 * np.conj(an) + bn1 * np.conj(bn)).real
        an1, bn1 = an, bn
        psi0, psi1, chi0, chi1 = psi1, psi, chi1, chi
        xi1 = complex(psi1, -chi1)
    return 2 / x ** 2 * qe, 2 / x ** 2 * qs, 2 * gs / qs


# sulfate: lognormal size distribution, effective radius 0.4 um, geometric standard deviation 1.5, refractive index 1.43
reff, sigma = 0.4, 1.5
r = np.logspace(-2.5, 0.8, 260)
n = np.exp(-0.5 * (np.log(r / (reff / np.exp(2.5 * np.log(sigma) ** 2))) / np.log(sigma)) ** 2)
optics = {}
for wl in (0.44, 0.55, 0.675, 0.87):
    q = np.array([mie(complex(1.43, 1e-8), 2 * np.pi * ri / wl) for ri in r])
    area = np.pi * r ** 2 * n
    optics[wl] = (np.trapezoid(q[:, 0] * area, np.log(r)), np.trapezoid(q[:, 2] * q[:, 1] * area, np.log(r))
                  / np.trapezoid(q[:, 1] * area, np.log(r)))
sulfate = {"ssa": 0.9999, "g": round(float(np.mean([optics[w][1] for w in (0.44, 0.55, 0.675)])), 3),
           "ae": round(float(-np.log(optics[0.44][0] / optics[0.87][0]) / np.log(0.44 / 0.87)), 3)}
sulfate

Out[3]: {'ssa': 0.9999, 'g': 0.75, 'ae': 0.498}


{'ssa': 0.9999, 'g': 0.75, 'ae': 0.498}

In [4]:
def uvspec(band, sza, aod, ssa, g, ae):
    w = "285 4000" if band == "BB" else "400 700"
    inp = f"""data_files_path {lrt}/data
atmosphere_file {lrt}/data/atmmod/afglms.dat
source solar {lrt}/data/solar_flux/NewGuey2003.dat
mol_abs_param reptran coarse
wavelength {w}
sza {sza}
altitude 0.3
albedo 0.2
rte_solver disort
number_of_streams 8
aerosol_default
aerosol_set_tau_at_wvl 550 {aod}
aerosol_angstrom {ae} {aod}
aerosol_modify ssa set {ssa}
aerosol_modify gg set {g}
output_process integrate
output_user edir edn
quiet
"""
    res = subprocess.run(shell + [f"cd {lrt}/bin && ./uvspec"], input=inp, capture_output=True, text=True).stdout.split()
    return round(float(res[0]) / 1000, 3), round(float(res[1]) / 1000, 3)


# added aerosol mixed with the clean sky aerosol, weighted by optical depth
tb = background["aod"]
runs = []
for case, a in (("smoke", smoke), ("sulfate", sulfate)):
    for x in aods:
        tau = tb + x
        ssa = (tb * background["ssa"] + x * a["ssa"]) / tau
        g = (tb * background["ssa"] * background["g"] + x * a["ssa"] * a["g"]) / (tau * ssa)
        ae = (tb * background["ae"] + x * a["ae"]) / tau
        for sza in szas:
            for band in ("PAR", "BB"):
                runs.append({"case": case, "x": x, "sza": sza, "band": band, "aod": round(tau, 4),
                             "ssa": round(min(ssa, 0.9999), 4), "g": round(g, 4), "ae": round(ae, 3)})
runs = pd.DataFrame(runs)
with ThreadPoolExecutor(8) as ex:
    res = list(ex.map(lambda r: uvspec(r.band, r.sza, r.aod, r.ssa, r.g, r.ae), runs.itertuples()))
runs[["edir", "edn"]] = res

In [5]:
# fraction of clean sky irradiance and change in diffuse fraction relative to no added aerosol
runs["tot"] = runs.edir + runs.edn
runs["DF"] = runs.edn / runs.tot
clean = runs[runs.x == 0].set_index(["case", "band", "sza"])
key = list(zip(runs.case, runs.band, runs.sza))
runs["frac"] = runs.tot / clean.loc[key, "tot"].values
runs["dDF"] = runs.DF - clean.loc[key, "DF"].values
runs[["case", "band", "sza", "x", "frac", "dDF"]].to_csv(out / "light_states.csv", index=False)
runs[(runs.band == "PAR") & (runs.sza == 30)].pivot(index="x", columns="case", values="frac").round(3)

Out[5]: 
case  smoke  sulfate
x                   
0.00  1.000    1.000
0.02  0.997    0.999
0.05  0.992    0.997
0.10  0.983    0.993
0.15  0.974    0.990
0.20  0.965    0.986
0.30  0.947    0.979
0.40  0.929    0.971
0.60  0.892    0.955
0.80  0.856    0.938
1.00  0.820    0.921
1.30  0.767    0.895
1.60  0.718    0.869
2.00  0.657    0.836


case,smoke,sulfate
x,,
0.00,1.000,1.000
0.02,0.997,0.999
0.05,0.992,0.997
0.10,0.983,0.993
0.15,0.974,0.990
0.20,0.965,0.986
0.30,0.947,0.979
0.40,0.929,0.971
0.60,0.892,0.955
